# Libraries

In [1]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_california_housing

from tensorflow.keras import layers, models
import tensorflow as tf

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# own functions
from active_regression import passive_sampling, uncertainty_sampling_reg, label_data, active_learning_reg, sasla_reg
from active_regression import tune_params, retrieve_params, cv_regression, create_regmodel

from time import time

# Models

## California Housing

In [2]:
california = fetch_california_housing(as_frame=True)
california_df = california.frame
california_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
X = california_df.drop(columns=['MedHouseVal']).to_numpy()
y = california_df.loc[:, 'MedHouseVal'].to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=51
)

In [5]:
acts = ['relu', 'leaky_relu', 'elu']
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='mse',
                              out_units=1, out_act='linear',
                              k=5, activation=activation,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/california/{activation}.json'
    )

[I 2026-09-25 07:37:05,168] A new study created in memory with name: no-name-595df385-cb03-48b6-b8c7-d5e3118a9b9b


[I 2026-09-25 07:41:21,891] Trial 0 finished with value: 0.3916826523948034 and parameters: {'hidden_layer': 21, 'learning_rate': 0.003625073723242179, 'momentum': 0.367834230369271, 'weight_decay': 2.5430405201770885e-05}. Best is trial 0 with value: 0.3916826523948034.
[I 2026-09-25 07:41:28,609] Trial 1 finished with value: inf and parameters: {'hidden_layer': 55, 'learning_rate': 0.0044267163631692774, 'momentum': 0.7719791575309446, 'weight_decay': 4.871099891275516e-05}. Best is trial 0 with value: 0.3916826523948034.
[I 2026-09-25 07:47:38,742] Trial 2 finished with value: 0.5027052022348896 and parameters: {'hidden_layer': 30, 'learning_rate': 0.00030453461462901546, 'momentum': 0.003086914225791774, 'weight_decay': 2.7796470503845208e-05}. Best is trial 0 with value: 0.3916826523948034.
[I 2026-09-25 07:51:41,132] Trial 3 finished with value: 0.35935990197524603 and parameters: {'hidden_layer': 80, 'learning_rate': 0.006751255703576771, 'momentum': 0.16903264886659605, 'weight

{'hidden_layer': 74, 'learning_rate': 0.0018151866586540834, 'momentum': 0.7973662107501709, 'weight_decay': 1.7969467048451465e-05}
0.3590877110713562


[I 2026-09-25 09:44:25,376] Trial 0 finished with value: 0.4570477371068279 and parameters: {'hidden_layer': 42, 'learning_rate': 0.00028896699348474427, 'momentum': 0.5612715360675683, 'weight_decay': 0.0014350985614902906, 'leaky_relu_alpha': 0.009315682944208856}. Best is trial 0 with value: 0.4570477371068279.
[I 2026-09-25 09:48:17,114] Trial 1 finished with value: inf and parameters: {'hidden_layer': 127, 'learning_rate': 0.001363307018331313, 'momentum': 0.7636265700459369, 'weight_decay': 0.0027334731715486305, 'leaky_relu_alpha': 0.11080990486570994}. Best is trial 0 with value: 0.4570477371068279.
[I 2026-09-25 09:54:29,270] Trial 2 finished with value: 0.3639014486574298 and parameters: {'hidden_layer': 123, 'learning_rate': 0.0031019592521037984, 'momentum': 0.13211481220152355, 'weight_decay': 1.2633197640445262e-05, 'leaky_relu_alpha': 0.0040559735830498255}. Best is trial 2 with value: 0.3639014486574298.
[I 2026-09-25 10:00:08,080] Trial 3 finished with value: 0.4500262

{'hidden_layer': 120, 'learning_rate': 0.005317746800160751, 'momentum': 0.20894003959285906, 'weight_decay': 2.9130934686171106e-06, 'leaky_relu_alpha': 0.0029763805914404997}
0.35092531902181173


[I 2026-09-25 13:39:50,793] Trial 0 finished with value: 0.4693701107610641 and parameters: {'hidden_layer': 47, 'learning_rate': 0.001734262365906389, 'momentum': 0.5047107896315642, 'weight_decay': 0.002032046171562881, 'elu_alpha': 1.3388134873784838}. Best is trial 0 with value: 0.4693701107610641.
[I 2026-09-25 13:47:39,141] Trial 1 finished with value: 0.453194784500096 and parameters: {'hidden_layer': 28, 'learning_rate': 0.0006927222559905066, 'momentum': 0.31742486722592045, 'weight_decay': 2.7946446306176854e-06, 'elu_alpha': 0.4791689934200174}. Best is trial 1 with value: 0.453194784500096.
[I 2026-09-25 13:53:07,686] Trial 2 finished with value: 0.47008506173751313 and parameters: {'hidden_layer': 96, 'learning_rate': 0.0027896345072476493, 'momentum': 0.09468942227907774, 'weight_decay': 3.285190188649579e-05, 'elu_alpha': 1.4476399735511372}. Best is trial 1 with value: 0.453194784500096.
[I 2026-09-25 13:53:16,936] Trial 3 finished with value: inf and parameters: {'hidd

{'hidden_layer': 82, 'learning_rate': 0.0004910762739354061, 'momentum': 0.9257687564245632, 'weight_decay': 1.3139548430197805e-05, 'elu_alpha': 0.1409809797548997}
0.3616198126481279


[I 2026-09-25 18:16:21,865] Trial 0 finished with value: 0.4070397459731634 and parameters: {'hidden_layer': 26, 'learning_rate': 0.00802892031446619, 'momentum': 0.06309611296163853, 'weight_decay': 8.20258637894404e-06}. Best is trial 0 with value: 0.4070397459731634.
[I 2026-09-25 18:20:58,206] Trial 1 finished with value: 0.4286748736933291 and parameters: {'hidden_layer': 74, 'learning_rate': 0.00019930292938529843, 'momentum': 0.9132897097333287, 'weight_decay': 5.658695845569574e-06}. Best is trial 0 with value: 0.4070397459731634.
[I 2026-09-25 18:26:07,019] Trial 2 finished with value: 0.5222480224578778 and parameters: {'hidden_layer': 22, 'learning_rate': 0.00027185651981566074, 'momentum': 0.11379155274446981, 'weight_decay': 7.047009018720604e-06}. Best is trial 0 with value: 0.4070397459731634.
[I 2026-09-25 18:31:22,458] Trial 3 finished with value: 0.47533748973298284 and parameters: {'hidden_layer': 15, 'learning_rate': 0.0008349020243737295, 'momentum': 0.013033356717

KeyboardInterrupt: 

In [10]:
acts = ['relu', 'leaky_relu', 'elu']
results = []
for activation in acts:
    setup = {'activation': activation}
    alpha = 0
    params = retrieve_params(f'params/california/{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_regression(X_pool, y_pool, out_units=1, out_act='linear', 
                    loss='mse', k=5, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'])
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/california/cv_validation.csv", index=False)

In [5]:
results_df = pd.read_csv("results/california/cv_validation.csv")
results_df

,activation,rmse,mape,R_squared
0,relu,0.6678,0.2625,0.6476
1,leaky_relu,0.5929,0.2447,0.7345
2,elu,0.6083,0.2558,0.7204


In [11]:
activation = 'leaky_relu'
params = retrieve_params(f'params/california/{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
model = create_regmodel(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'], 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='mse')
# params['learning_rate']
history = []
runs = 100
# runs = 200
epochs = 200 // runs

fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_pred = model(X_val, training=False).numpy().ravel()

    if not np.all(np.isfinite(y_pred)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break

    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    toc = time()

    iteration['Training loss'] = train_loss[-1]
    iteration['RMSE'] = rmse
    iteration['MAPE'] = mape
    iteration['R2-score'] = r2
    iteration['time'] = toc - tic
    history.append(iteration)
    
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/california/passive_learning.csv", index=False)

Iteration 1: 
	0.8864449858665466
	0.5983476042747498
	0.5244823098182678
	0.4976836144924164
Iteration 2: 
	0.4819352626800537
	0.4785275161266327
	0.476483553647995
	0.4542883634567261
Iteration 3: 
	0.44996973872184753
	0.4482862651348114
	0.4493209719657898
	0.4393377900123596
Iteration 4: 
	0.4342212975025177
	0.42445966601371765
	0.4208027422428131
	0.4181661605834961
Iteration 5: 
	0.4150742292404175
	0.4157498776912689
	0.40638625621795654
	0.40310871601104736
Iteration 6: 
	0.4062226712703705
	0.45524150133132935
	0.4911082983016968
	0.4327504634857178
Iteration 7: 
	0.42066067457199097
	0.4433033764362335
	0.41511794924736023
	0.4099682867527008
Iteration 8: 
	0.40156289935112
	0.39395928382873535
	0.39474862813949585
	0.3922727108001709
Iteration 9: 
	0.38521888852119446
	0.3894878327846527
	0.3858407139778137
	0.3829437792301178
Iteration 10: 
	0.384768009185791
	0.3820318579673767
	0.3764415681362152
	0.37688153982162476
Iteration 11: 
	0.3764477074146271
	0.38493111729621

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/california/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_reg(X_train, y_pool, X_val, y_test, uncertainty_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'],
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/california/uncertainty_sampling.csv", index=False)

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/california/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9, 0.8, 0.7, 0.6]:
    history = sasla_reg(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=32, activation=activation, alpha=params['leaky_relu_alpha'],
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/california/sasla_{beta}.csv", index=False)

## Concrete

In [11]:
concrete_compressive_strength = fetch_ucirepo(id=165) 
ccs_df = pd.concat(
    [concrete_compressive_strength.data.features, concrete_compressive_strength.data.targets],
    axis=1
)
ccs_df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Concrete compressive strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [12]:
X = concrete_compressive_strength.data.features.to_numpy()
y = concrete_compressive_strength.data.targets.to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu']
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='mse',
                              out_units=1, out_act='linear',
                              k=5, activation=activation,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/concrete/{activation}.json'
    )

In [13]:
acts = ['relu', 'leaky_relu', 'elu']
results = []
for activation in acts:
    setup = {'activation': activation}
    alpha = 0
    params = retrieve_params(f'params/concrete/{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_regression(X_pool, y_pool, out_units=1, out_act='linear', 
                    loss='mse', k=5, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'])
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/concrete/cv_validation.csv", index=False)

In [ ]:
results_df = pd.read_csv("results/concrete/cv_validation.csv")
results_df

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/concrete/{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
model = create_regmodel(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'], 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='mse')
# params['learning_rate']
history = []
runs = 100
# runs = 200
epochs = 200 // runs

fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_pred = model(X_val, training=False).numpy().ravel()

    if not np.all(np.isfinite(y_pred)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break

    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    toc = time()

    iteration['Training loss'] = train_loss[-1]
    iteration['RMSE'] = rmse
    iteration['MAPE'] = mape
    iteration['R2-score'] = r2
    iteration['time'] = toc - tic
    history.append(iteration)
    
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/concrete/passive_learning.csv", index=False)

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/concrete/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_reg(X_train, y_pool, X_val, y_test, uncertainty_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'],
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/concrete/uncertainty_sampling.csv", index=False)

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/concrete/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9, 0.8, 0.7, 0.6]:
    history = sasla_reg(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=32, activation=activation, alpha=params['leaky_relu_alpha'],
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/concrete/sasla_{beta}.csv", index=False)

## Seoul Bike sharing

In [14]:
seoul_bike_sharing_demand = fetch_ucirepo(id=560)
sbsd_df = pd.concat(
    [seoul_bike_sharing_demand.data.features, seoul_bike_sharing_demand.data.targets],
    axis=1
)

drop = ['Date', 'Hour', "Month", "DayOfWeek", 'Seasons']
sbsd_df['Date'] = pd.to_datetime(
    sbsd_df["Date"],
    format="%d/%m/%Y"
)
sbsd_df["Hour_sin"] = np.sin(2 * np.pi * sbsd_df["Hour"] / 24)
sbsd_df["Hour_cos"] = np.cos(2 * np.pi * sbsd_df["Hour"] / 24)

sbsd_df["Month"] = sbsd_df["Date"].dt.month
sbsd_df["DayOfWeek"] = sbsd_df["Date"].dt.dayofweek

sbsd_df["Month_sin"] = np.sin(2 * np.pi * sbsd_df["Month"] / 12)
sbsd_df["Month_cos"] = np.cos(2 * np.pi * sbsd_df["Month"] / 12)

sbsd_df["DayOfWeek_sin"] = np.sin(2 * np.pi * sbsd_df["DayOfWeek"] / 7)
sbsd_df["DayOfWeek_cos"] = np.cos(2 * np.pi * sbsd_df["DayOfWeek"] / 7)

sbsd_df["Holiday"] = (sbsd_df["Holiday"] == "Holiday").astype(int)
sbsd_df["Functioning Day"] = (sbsd_df["Functioning Day"] == "Yes").astype(int)

sbsd_df.drop(columns=drop, inplace=True)

sbsd_df.head()

,Rented Bike Count,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Holiday,Functioning Day,Hour_sin,Hour_cos,Month_sin,Month_cos,DayOfWeek_sin,DayOfWeek_cos
0,254,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,0,1,0.000000,1.000000,-2.449294e-16,1.0,-0.433884,-0.900969
1,204,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,0,1,0.258819,0.965926,-2.449294e-16,1.0,-0.433884,-0.900969
2,173,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,0,1,0.500000,0.866025,-2.449294e-16,1.0,-0.433884,-0.900969
3,107,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,0,1,0.707107,0.707107,-2.449294e-16,1.0,-0.433884,-0.900969
4,78,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,0,1,0.866025,0.500000,-2.449294e-16,1.0,-0.433884,-0.900969


In [18]:
X = sbsd_df.drop(columns=['Rented Bike Count']).to_numpy()
y = sbsd_df['Rented Bike Count'].to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu']
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='mse',
                              out_units=1, out_act='linear',
                              k=5, activation=activation,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/seoul_bikes/{activation}.json'
    )

In [19]:
acts = ['relu', 'leaky_relu', 'elu']
results = []
for activation in acts:
    setup = {'activation': activation}
    alpha = 0
    params = retrieve_params(f'params/seoul_bikes/{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_regression(X_pool, y_pool, out_units=1, out_act='linear', 
                    loss='mse', k=5, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'])
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/seoul_bikes/cv_validation.csv", index=False)

In [ ]:
results_df = pd.read_csv("results/seoul_bikes/cv_validation.csv")
results_df

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/seoul_bikes/{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
model = create_regmodel(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'], 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='mse')
# params['learning_rate']
history = []
runs = 100
# runs = 200
epochs = 200 // runs

fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_pred = model(X_val, training=False).numpy().ravel()

    if not np.all(np.isfinite(y_pred)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break

    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    toc = time()

    iteration['Training loss'] = train_loss[-1]
    iteration['RMSE'] = rmse
    iteration['MAPE'] = mape
    iteration['R2-score'] = r2
    iteration['time'] = toc - tic
    history.append(iteration)
    
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/seoul_bikes/passive_learning.csv", index=False)

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/seoul_bikes/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_reg(X_train, y_pool, X_val, y_test, uncertainty_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'],
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/seoul_bikes/uncertainty_sampling.csv", index=False)

In [ ]:
activation = 'leaky_relu'
params = retrieve_params(f'params/seoul_bikes/{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9, 0.8, 0.7, 0.6]:
    history = sasla_reg(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=32, activation=activation, alpha=params['leaky_relu_alpha'],
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/seoul_bikes/sasla_{beta}.csv", index=False)